<a href="https://colab.research.google.com/github/beingAnujChaudhary/Practical-Statistics-for-DS/blob/main/chapter_04_regression_and_prediction/exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/Practical-Statistics-for-DS.git

# Move into repository
%cd /content/Practical-Statistics-for-DS

# Move into Chapter 4 folder
%cd chapter_04_regression_and_prediction

Mounted at /content/drive
Cloning into 'Practical-Statistics-for-DS'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 176 (delta 107), reused 69 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (176/176), 596.21 KiB | 6.70 MiB/s, done.
Resolving deltas: 100% (107/107), done.
/content/Practical-Statistics-for-DS
/content/Practical-Statistics-for-DS/chapter_04_regression_and_prediction


# Chapter 04 Exercises: Regression and Prediction

> Source: *Practical Statistics for Data Scientists, 2nd Edition*  
> Chapter 4: Regression and Prediction

## Goals

In this notebook, I will practise:
- Simple regression
- Multiple regression
- Prediction
- Coefficient interpretation
- Model evaluation (R², RMSE, Cross-Validation)
- Residual thinking
- Categorical variables and dummy encoding
- Interaction effects
- Confounding variables
- Regression diagnostics
- Nonlinear relationships

---

## 📋 Instructions

- Attempt each exercise before viewing the solution
- Focus on the interpretation of the coefficients and diagnostics, not just the code
- Explaining why a model behaves a certain way is just as important as building it
- Solutions are provided in collapsed sections—expand only after attempting the problem

---

## Setup

<details>
<summary>Click to view solution</summary>

```python
# Imports
try:
    from utils.notebook_setup import *
except:
    import warnings
    warnings.filterwarnings("ignore")
    
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import os
    
    plt.style.use("seaborn-v0_8")
    np.random.seed(42)

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, KFold
from statsmodels.stats.outliers_influence import OLSInfluence

# Load King County Housing Data
local_path = '../datasets/raw/kc_house_data.csv'
url = 'https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/kc_house_data.csv'

if os.path.exists(local_path):
    house = pd.read_csv(local_path)
else:
    print("Downloading King County Housing data...")
    house = pd.read_csv(url)

# Safeguard: Map 'price' to 'AdjSalePrice' if needed
if 'AdjSalePrice' not in house.columns and 'price' in house.columns:
    house['AdjSalePrice'] = house['price']

# Create a smaller subset for faster diagnostics
np.random.seed(42)
house_sample = house.sample(5000, random_state=42).copy()

print(f"Dataset loaded. Shape: {house.shape}")
print(f"Sample shape: {house_sample.shape}")
```
</details>



---

# Exercise 1: Simple Linear Regression & Interpretation

## Scenario

You want to predict house price using house size.

Question: Do larger houses cost more?

<details>
<summary>Click to view solution</summary>

```python
# Create simulated data
house_size = np.random.normal(1500, 300, 200)
house_price = house_size * 200 + np.random.normal(0, 30000, 200)

housing = pd.DataFrame({
    "House_Size": house_size,
    "House_Price": house_price
})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=housing, x="House_Size", y="House_Price")
plt.title("House Size vs Price")
plt.show()
```
</details>

## Reflection

Questions:
1. Is relationship visible?
2. Does it appear linear?
3. Why is there variation?


---

# Exercise 2: Fit Simple Linear Regression

<details>
<summary>Click to view solution</summary>

```python
X = housing[["House_Size"]]
y = housing["House_Price"]

model = LinearRegression()
model.fit(X, y)

print(f"Intercept: {model.intercept_:.2f}")
print(f"Coefficient: {model.coef_[0]:.2f}")
print(f"\nInterpretation: For every additional sq ft, price increases by ${model.coef_[0]:.0f}")
```
</details>

## Reflection

Questions:
1. What does coefficient mean?
2. Does bigger house size increase price?
3. What does intercept represent?



---

# Exercise 3: Fit Simple Linear Regression with Housing Data

## Context

Fit a simple linear regression model predicting `AdjSalePrice` using only `SqFtTotLiving`.

<details>
<summary>Click to view solution</summary>

```python
# 1. Fit the model
model_simple = smf.ols('AdjSalePrice ~ SqFtTotLiving', data=house).fit()

# 2. Calculate Metrics
y_pred = model_simple.fittedvalues
rmse = np.sqrt(mean_squared_error(house['AdjSalePrice'], y_pred))
r2 = r2_score(house['AdjSalePrice'], y_pred)

print(f"In-sample RMSE: ${rmse:,.0f}")
print(f"In-sample R²: {r2:.4f}")

# 3. Interpretation
slope = model_simple.params['SqFtTotLiving']
intercept = model_simple.params['Intercept']
print(f"\nInterpretation: The base price is roughly ${intercept:,.0f}.")
print(f"For every additional square foot, price increases by ${slope:,.0f}.")

# 4. Visualization
plt.figure(figsize=(10, 6))
plot_data = house.sample(2000, random_state=42)
sns.scatterplot(data=plot_data, x='SqFtTotLiving', y='AdjSalePrice', alpha=0.3, color='blue')
plt.plot(plot_data['SqFtTotLiving'], model_simple.predict(plot_data), color='red', linewidth=2, label='OLS Fit')
plt.xlabel('SqFtTotLiving')
plt.ylabel('AdjSalePrice')
plt.title('Simple Linear Regression: Price vs. Living Space')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```
</details>

**ML Relevance Insight**: Notice the "funnel" shape in the scatterplot. The variance of prices increases as house size increases. This is a classic sign of **heteroskedasticity**, which violates OLS assumptions and suggests a log-transformation might be beneficial.


---

# Exercise 4: Visualise Regression Line

<details>
<summary>Click to view solution</summary>

```python
predictions = model.predict(X)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=housing, x="House_Size", y="House_Price")
plt.plot(housing["House_Size"], predictions, color='red')
plt.title("Regression Line")
plt.show()
```
</details>

## Reflection

Questions:
1. Does line fit data well?
2. Why is fit imperfect?
3. What causes prediction error?


---

# Exercise 5: Prediction

## Task

Predict price for a 2000 sq ft house.

<details>
<summary>Click to view solution</summary>

```python
new_house = pd.DataFrame({"House_Size": [2000]})
predicted_price = model.predict(new_house)

print(f"Predicted Price: ${predicted_price[0]:,.2f}")
print(f"\nUsing formula: {model.intercept_:.0f} + {model.coef_[0]:.0f} × 2000 = {model.intercept_ + model.coef_[0] * 2000:,.0f}")
```
</details>

## Reflection

Questions:
1. Does prediction seem reasonable?
2. Why are predictions uncertain?
3. What factors are missing?


---

# Exercise 6: Model Evaluation

## Task

Evaluate model using R² score and Mean Squared Error.

<details>
<summary>Click to view solution</summary>

```python
r2 = r2_score(y, predictions)
mse = mean_squared_error(y, predictions)

print(f"R² Score: {r2:.4f}")
print(f"Mean Squared Error: {mse:,.2f}")
print(f"RMSE: {np.sqrt(mse):,.2f}")
print(f"\nR² of {r2:.4f} means {r2*100:.1f}% of variance is explained.")
print(f"RMSE represents the typical prediction error.")
```
</details>

## Reflection

Questions:
1. Was R² high?
2. Is high R² always good?
3. Why does MSE matter?


---

# Exercise 7: Multiple Regression

## Scenario

Predict house price using house size, bedrooms, and house age.

<details>
<summary>Click to view solution</summary>

```python
# Create simulated data
housing_data = pd.DataFrame({
    "House_Size": np.random.normal(1500, 300, 300),
    "Bedrooms": np.random.randint(1, 6, 300),
    "House_Age": np.random.randint(1, 50, 300)
})

housing_data["House_Price"] = (
    housing_data["House_Size"] * 200 +
    housing_data["Bedrooms"] * 10000 -
    housing_data["House_Age"] * 800 +
    np.random.normal(0, 30000, 300)
)

X_multi = housing_data[["House_Size", "Bedrooms", "House_Age"]]
y_multi = housing_data["House_Price"]

multiple_model = LinearRegression()
multiple_model.fit(X_multi, y_multi)

coefficients = pd.DataFrame({
    "Feature": X_multi.columns,
    "Coefficient": multiple_model.coef_
})
print(coefficients)
print(f"\nIntercept: {multiple_model.intercept_:.2f}")
```
</details>

<details>
<summary>Click to view solution</summary>

```python
# Evaluate multiple regression
multiple_predictions = multiple_model.predict(X_multi)
r2_multi = r2_score(y_multi, multiple_predictions)

print(f"R² Score: {r2_multi:.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_multi, multiple_predictions)):.2f}")
```
</details>

## Reflection

Questions:
1. Which feature matters most?
2. Why is house age negative?
3. Why are coefficients useful?


---

# Exercise 8: Compare Simple vs Multiple Regression

## Question

Which model performs better? Compare simple regression vs multiple regression.

## Reflection

Questions:
1. Which model had higher R²?
2. Why did extra variables help?
3. Can too many variables become dangerous?


---

# Exercise 8: Compare Simple vs Multiple Regression

## Question

Which model performs better? Compare simple regression vs multiple regression.

## Reflection

Questions:
1. Which model had higher R²?
2. Why did extra variables help?
3. Can too many variables become dangerous?


---

# Exercise 9: Multiple Regression & Cross-Validation

## Context

In-sample metrics like R² are overly optimistic. Cross-Validation gives a realistic estimate of how the model will perform on unseen data.

<details>
<summary>Click to view solution</summary>

```python
predictors = ['SqFtTotLiving', 'SqFtLot', 'Bathrooms', 'Bedrooms', 'BldgGrade']
outcome = 'AdjSalePrice'

# Drop NAs for sklearn
df_clean = house[predictors + [outcome]].dropna()
X_cv = df_clean[predictors]
y_cv = df_clean[outcome]

# In-sample RMSE
sk_model = LinearRegression()
sk_model.fit(X_cv, y_cv)
y_pred_in = sk_model.predict(X_cv)
rmse_in = np.sqrt(mean_squared_error(y_cv, y_pred_in))
print(f"In-sample RMSE: ${rmse_in:,.0f}")

# 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(sk_model, X_cv, y_cv, cv=kf, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)

print(f"CV RMSE scores: {cv_rmse}")
print(f"Mean Out-of-sample CV RMSE: ${cv_rmse.mean():,.0f} ± ${cv_rmse.std():,.0f}")

# Comparison
gap = cv_rmse.mean() - rmse_in
print(f"\nGap between CV and In-sample RMSE: ${gap:,.0f}")
print("This gap represents overfitting to the training data.")
```
</details>

**ML Relevance Insight**: The out-of-sample RMSE is almost always higher than the in-sample RMSE. If this gap is massive, your model is memorizing noise rather than learning signal.


---

# Exercise 10: Categorical Variables & Dummy Encoding

## Problem

Regression expects numbers, but real-world data contains categories (cabin type, airline, customer segment).

<details>
<summary>Click to view solution</summary>

```python
# Create example airline data
airline_df = pd.DataFrame({
    "Cabin": np.random.choice(["economy", "business", "first"], size=200),
    "Spend": np.random.normal(300, 50, 200)
})

print("Original data:")
print(airline_df.head())

# Dummy encoding
encoded_df = pd.get_dummies(airline_df, columns=["Cabin"], drop_first=True)
print("\nEncoded data:")
print(encoded_df.head())

# Fit model
X_cat = encoded_df.drop("Spend", axis=1)
y_cat = encoded_df["Spend"]

cabin_model = LinearRegression()
cabin_model.fit(X_cat, y_cat)

coef_df = pd.DataFrame({
    "Feature": X_cat.columns,
    "Coefficient": cabin_model.coef_
})
print("\nCoefficients:")
print(coef_df)
```
</details>

## Reflection

Questions:
1. What does positive coefficient mean?
2. Why must categories become numbers?
3. Why remove one category (drop_first=True)?


---

# Exercise 11: Factor Variables & Confounding

## Context

Omitting important variables (like Location) can cause **confounding**, where coefficients become biased or even flip signs.

<details>
<summary>Click to view solution</summary>

```python
# 1. Base Model (without location)
model_base = smf.ols('AdjSalePrice ~ SqFtTotLiving + Bathrooms + Bedrooms + BldgGrade', data=house).fit()
print("Base Model Coefficients:")
print(model_base.params[['Bedrooms', 'Bathrooms', 'SqFtTotLiving']].round(2))

# 2. Create ZipGroup based on median price per zip
zip_medians = house.groupby('ZipCode')['AdjSalePrice'].median()
house['ZipMedianPrice'] = house['ZipCode'].map(zip_medians)
house['ZipGroup'] = pd.qcut(house['ZipMedianPrice'].dropna(), q=5, labels=False, duplicates='drop')

# 3. Re-fit model with Location
model_conf = smf.ols('AdjSalePrice ~ SqFtTotLiving + Bathrooms + Bedrooms + BldgGrade + C(ZipGroup)',
                     data=house.dropna(subset=['ZipGroup'])).fit()

print("\nModel with ZipGroup Coefficients:")
print(model_conf.params[['Bedrooms', 'Bathrooms', 'SqFtTotLiving']].round(2))
print("\nNotice how coefficients change when location is included!")
```
</details>

**ML Relevance Insight**: In the base model, `Bedrooms` may have a negative coefficient. Adding `ZipGroup` controls for location, revealing the true partial effect. Omitting location mixes cheap rural mansions with expensive urban condos, creating a spurious relationship.


---

# Exercise 12: Interaction Effects

## Key idea

Sometimes one variable changes the effect of another. Example: Study hours may matter differently for beginners vs experts.

<details>
<summary>Click to view solution</summary>

```python
# Create student data with interaction
student_df = pd.DataFrame({
    "Study_Hours": np.random.randint(1, 10, 300),
    "Experience": np.random.randint(0, 2, 300)
})

student_df["Score"] = (
    student_df["Study_Hours"] * 5 +
    student_df["Experience"] * 10 +
    (student_df["Study_Hours"] * student_df["Experience"] * 2) +
    np.random.normal(0, 5, 300)
)

# Create interaction term
student_df["Interaction"] = student_df["Study_Hours"] * student_df["Experience"]

X_int = student_df[["Study_Hours", "Experience", "Interaction"]]
y_int = student_df["Score"]

interaction_model = LinearRegression()
interaction_model.fit(X_int, y_int)

coef_int = pd.DataFrame({
    "Feature": X_int.columns,
    "Coefficient": interaction_model.coef_
})
print(coef_int)
print(f"\nIntercept: {interaction_model.intercept_:.2f}")
print("\nThe interaction term shows how experience amplifies the effect of study hours.")
```
</details>

## Reflection

Questions:
1. Did experience change study effect?
2. Why do interactions matter?
3. Can business behaviour depend on context?



---

# Exercise 13: Confounding Variables Demonstration

<details>
<summary>Click to view solution</summary>

```python
# Generate confounding example
temperature = np.random.normal(30, 5, 200)
ice_cream_sales = temperature * 10 + np.random.normal(0, 5, 200)
swimming = temperature * 8 + np.random.normal(0, 5, 200)

confounding_df = pd.DataFrame({
    "Temperature": temperature,
    "Ice_Cream": ice_cream_sales,
    "Swimming": swimming
})

print("Correlation matrix:")
print(confounding_df.corr().round(3))
print("\nIce cream and swimming are correlated (r > 0.9), but temperature is the true cause.")
print("This is a classic confounding relationship.")
```
</details>

## Reflection

Questions:
1. Why are variables correlated?
2. Does ice cream cause swimming?
3. Why are confounders dangerous?



---

# Exercise 14: Regression Diagnostics (Outliers & Influence)

## Context

A single anomalous record can drastically pull the OLS regression line, ruining predictions for everyone else.

<details>
<summary>Click to view solution</summary>

```python
# 1. Fit model on sample
model_diag = smf.ols('AdjSalePrice ~ SqFtTotLiving + BldgGrade', data=house_sample).fit()

# 2. Calculate Diagnostics
influence = OLSInfluence(model_diag)
std_resid = influence.resid_studentized_internal
cooks_d = influence.cooks_distance[0]
hat_values = influence.hat_matrix_diag

# 3. Influence Plot
threshold = 4 / len(house_sample)

plt.figure(figsize=(10, 6))
sizes = 1000 * np.sqrt(cooks_d)
plt.scatter(hat_values, std_resid, s=sizes, alpha=0.5, edgecolors='black', color='skyblue')

extreme_mask = cooks_d > threshold
plt.scatter(hat_values[extreme_mask], std_resid[extreme_mask], s=sizes[extreme_mask],
            color='red', edgecolors='black', label='High Influence')

plt.axhline(-2.5, linestyle='--', color='gray')
plt.axhline(2.5, linestyle='--', color='gray')
plt.xlabel('Leverage (Hat-Values)')
plt.ylabel('Standardized Residuals')
plt.title('Influence Plot (Size = Cook\'s Distance)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 4. Isolate the worst offender
worst_idx = np.argmax(cooks_d)
print(f"Most influential record index: {worst_idx}")
print(f"Cook's Distance: {cooks_d[worst_idx]:.4f} (Threshold: {threshold:.4f})")
display(house_sample.iloc[[worst_idx]][['AdjSalePrice', 'SqFtTotLiving', 'BldgGrade']])
```
</details>

**ML Relevance Insight**: Cook's Distance captures the dangerous intersection of high leverage and massive errors. In production ML, filter these out or cap target variables (winsorization) before training.



---

# Exercise 15: Residual Analysis

<details>
<summary>Click to view solution</summary>

```python
# Residual plot for simple model
residuals = y - predictions

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(predictions, residuals, alpha=0.5)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel("Predicted Values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residual Plot (Check for patterns)")

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel("Residuals")
axes[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()
```
</details>

## Reflection

Questions:
1. Are residuals random?
2. Do you see patterns?
3. Why should residuals be random?



---

# Exercise 16: Capturing Nonlinearity (Polynomials & Splines)

## Context

The relationship between Square Footage and Price is rarely perfectly linear.

<details>
<summary>Click to view solution</summary>

```python
# Use a subset for cleaner plotting
df_viz = house.sample(3000, random_state=42).copy()

# Base Linear Model for Partial Residuals
model_lin = smf.ols('AdjSalePrice ~ SqFtTotLiving + BldgGrade', data=df_viz).fit()
partial_resid = model_lin.resid + (model_lin.params['SqFtTotLiving'] * df_viz['SqFtTotLiving'])

# Polynomial Model (Degree 2)
model_poly = smf.ols('AdjSalePrice ~ SqFtTotLiving + I(SqFtTotLiving**2) + BldgGrade', data=df_viz).fit()

# Spline Model (B-Splines)
formula_spline = 'AdjSalePrice ~ bs(SqFtTotLiving, df=5) + BldgGrade'
model_spline = smf.ols(formula_spline, data=df_viz).fit()

# Visualization
plt.figure(figsize=(12, 7))
plt.scatter(df_viz['SqFtTotLiving'], partial_resid, alpha=0.2, color='gray', label='Partial Residuals')
plt.plot(df_viz['SqFtTotLiving'].sort_values(),
         model_poly.fittedvalues.loc[df_viz['SqFtTotLiving'].sort_values().index],
         color='red', label='Polynomial (Deg 2)')
plt.plot(df_viz['SqFtTotLiving'].sort_values(),
         model_spline.fittedvalues.loc[df_viz['SqFtTotLiving'].sort_values().index],
         color='green', label='Spline (df=5)')
plt.xlabel('SqFtTotLiving')
plt.ylabel('Partial Residual (Price Impact)')
plt.title('Capturing Nonlinearity: Polynomial vs. Spline')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```
</details>

**ML Relevance Insight**: Polynomials are easy but can go wildly out of bounds at edges. Splines are piecewise and localized, making them safer. Tree-based models (XGBoost/Random Forest) automatically capture these non-linearities without manual feature engineering.


---

# Exercise 17: Synthesis & Critical Thinking

**Answer in your own words:**

### 1. The Dummy Variable Trap

If you have a categorical variable `DayOfWeek` (7 levels) and you use One-Hot Encoding to create 7 binary columns, what will happen if you pass all 7 columns into OLS along with an intercept? How does `scikit-learn` handle this differently?

<details>
<summary>Click to view solution</summary>

```python
answer_1 = """
The Dummy Variable Trap:
Passing all 7 columns plus an intercept causes perfect multicollinearity
(the sum of the 7 dummy columns equals the intercept column of 1s).

The matrix (X^T X) becomes non-invertible, and statsmodels will either
throw an error or drop a variable automatically.

scikit-learn doesn't calculate p-values or invert matrices in the same way
(it uses gradient descent/SVD), so it won't crash, but the coefficients
will be unstable and uninterpretable.

Best practice: Always drop one category (reference level) when using linear models.
"""
print(answer_1)
```
</details>

### 2. Extrapolation Danger

Your model was trained on houses between 1,000 and 4,000 sq ft. A user inputs a 15,000 sq ft mansion. The model predicts a negative price. Mathematically, why did this happen, and how would you prevent it in a production API?

<details>
<summary>Click to view solution</summary>

```python
answer_2 = """
Extrapolation Danger:
Linear models assume the trend continues infinitely. If the slope is negative
(or if a polynomial curves downward), it will eventually cross the x-axis
into negative numbers.

Prevention in Production:
- Never trust linear models outside the training domain
- Implement clipping/capping in your API: min(max(prediction, 0), max_train_price)
- Use tree-based models which naturally "flatten out" at training data boundaries
- Add input validation to warn when inputs are outside training range
"""
print(answer_2)
```
</details>

### 3. RMSE vs. MAE

The book focuses heavily on RMSE. In what business scenario would you prefer to optimize for MAE (Mean Absolute Error) instead?

<details>
<summary>Click to view solution</summary>

```python
answer_3 = """
RMSE vs. MAE:
RMSE squares errors before averaging, heavily penalizing large outliers.

Use MAE when:
- Business cost scales linearly with error (e.g., paying $10 fee for every
  $100 you're off in inventory forecasting)
- You want a metric in the same units as the target
- You don't want outliers to dominate the optimization

Use RMSE when:
- Large errors are catastrophically worse than small ones (e.g., server load
  capacity planning where under-prediction causes total crash)
- You want to penalize large deviations heavily
"""
print(answer_3)
```
</details>


---

# Mini Exercise

## Question

Suppose R² = 0.95. Does this guarantee good prediction? Could the model still fail? What risks exist?

---

# ML Relevance Exercise

## Question

Why does Chapter 4 matter in Machine Learning?

Think about:
- Baseline models
- Feature importance
- Prediction vs explanation
- Model evaluation
- Residual analysis
- Overfitting detection

---

# Interview Questions

1. **What is linear regression?**
2. **What does R² tell us?**
3. **Difference between simple and multiple regression?**
4. **What are dummy variables?**
5. **Why avoid dummy variable trap?**
6. **What is interaction effect?**
7. **What is confounding?**
8. **Why do we need cross-validation?**
9. **What is RMSE?**
10. **Why check residuals?**
11. **What is heteroskedasticity?**
12. **Why might high R² be misleading?**
13. **What is Cook's Distance and why does it matter?**
14. **When would you use polynomial regression vs splines?**

---

# Challenge Problems

1. Build regression model with interaction terms
2. Compare polynomial vs spline regression
3. Create residual diagnostic plots
4. Handle categorical variables with many levels
5. Perform cross-validation with different K values
6. Identify and handle influential observations
7. Compare RMSE vs MAE optimization

---

# Progress Checklist

- [ ] Built simple regression
- [ ] Interpreted coefficients
- [ ] Made predictions
- [ ] Evaluated models (R², RMSE)
- [ ] Built multiple regression
- [ ] Compared simple vs multiple regression
- [ ] Performed cross-validation
- [ ] Handled categorical variables
- [ ] Modeled interaction effects
- [ ] Identified confounding variables
- [ ] Analysed residuals
- [ ] Created influence plots (Cook's Distance)
- [ ] Explored nonlinear relationships (polynomials and splines)
- [ ] Completed critical thinking questions
- [ ] Connected concepts to ML
- [ ] Ready for Chapter 5

---

> **Pro Tip**: The `OLSInfluence` class from `statsmodels` is a goldmine for data cleaning. Before feeding data into a complex Neural Network or XGBoost model, running a quick OLS regression and filtering out records with massive Cook's Distances can significantly improve your downstream model's stability!
